# AI Invoice Processing: run the Claude extraction step

This notebook runs the one step that needs an API key: Claude reads the 836 invoice PDFs. Everything else (the PDFs, the rules + OCR results, the matching code) is already in the project zip.

**Before you start:** add your key in Colab's Secrets panel (key icon on the left) as `ANTHROPIC_API_KEY`, and turn on notebook access.

## 1. Upload and unzip the project

In [ ]:
from google.colab import files
up = files.upload()  # choose AI_Invoice_Processing.zip
!unzip -q -o AI_Invoice_Processing.zip -d /content
%cd /content/AI_Invoice_Processing

## 2. Install packages

In [ ]:
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install -r requirements.txt

## 3. Load the API key

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('key loaded')

## 4. Test on 10 invoices first (about 5 cents)

In [ ]:
!python src/extract_llm.py --limit 10

## 5. Run all 836 invoices (about $4 on Claude Haiku 4.5)
Results are cached, so if it stops you can run this cell again and it picks up where it left off.

In [ ]:
!python src/extract_llm.py

## 6. Score all methods and see the results

In [ ]:
!python src/process.py
import pandas as pd
pd.set_option('display.max_columns', 20)
summary = pd.read_csv('output/summary_by_method.csv', index_col=0)
summary.T

In [ ]:
acc = pd.read_csv('output/field_accuracy.csv')
acc.round(3)

## 7. Download the results

In [ ]:
!zip -q -r results.zip output/summary_by_method.csv output/field_accuracy.csv output/invoice_decisions.csv output/queue_simulation.csv output/extracted_llm.csv output/llm_cache
files.download('results.zip')